# Semana 34 · ¿Por qué brilla el Sol?

Este cuaderno acompaña a la sesión de proyecto de la semana 34: el balance de energía del Sol, sus escalas de tiempo, el camino aleatorio de la luz en su interior y el equilibrio hidrostático.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El balance de energía

Constantes y datos del Sol, y las cuentas de la sesión 1: masa perdida por segundo, defecto de masa, fusiones y neutrinos por segundo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G, c, kB = 6.674e-11, 2.998e8, 1.381e-23
m_p, u_kg, MeV = 1.6726e-27, 1.66054e-27, 1.602e-13
M_sol, R_sol, L_sol = 1.989e30, 6.957e8, 3.828e26
ANO = 3.156e7
UA_cm = 1.496e13

dm_dt = L_sol / c**2
print(f"masa convertida en energía: {dm_dt:.2e} kg/s ({dm_dt / 1000:.1e} toneladas por segundo)")

m_H, m_He = 1.007825, 4.002603           # masas atómicas en u
defecto = 4 * m_H - m_He
E_fusion = defecto * 931.494             # MeV
print(f"defecto de masa: {defecto:.6f} u = {100 * defecto / (4 * m_H):.2f} % → {E_fusion:.2f} MeV por helio")

fusiones = L_sol / (E_fusion * MeV)
neutrinos = 2 * fusiones
flujo_nu = neutrinos / (4 * np.pi * UA_cm**2)
print(f"fusiones por segundo: {fusiones:.2e}")
print(f"neutrinos por cm² y segundo en la Tierra: {flujo_nu:.2e}")
print(f"hidrógeno consumido: {4 * m_H * u_kg * fusiones:.2e} kg/s")

Fíjate en la última línea: el Sol funde unos 600 millones de toneladas de hidrógeno por segundo y, de ellas, convierte en energía unos 4 millones. El resto queda como helio.

## 2. Tres escalas de tiempo

In [ ]:
E_quimica = 3e7 * M_sol                          # carbón: 3e7 J/kg
E_grav = G * M_sol**2 / R_sol                    # orden de magnitud
E_nuclear = 0.007 * 0.1 * M_sol * c**2           # 0,7 % de la masa del 10 % central

for nombre, E in [("química (carbón)", E_quimica), ("gravitatoria (Kelvin-Helmholtz)", E_grav), ("nuclear", E_nuclear)]:
    t = E / L_sol / ANO
    print(f"{nombre:<32} {t:10.2e} años")

In [ ]:
nombres = ["química", "gravitatoria", "nuclear", "edad de la Tierra", "edad del universo"]
tiempos = [E_quimica / L_sol / ANO, E_grav / L_sol / ANO, E_nuclear / L_sol / ANO, 4.5e9, 1.38e10]
colores = ["tab:gray", "tab:orange", "tab:red", "tab:green", "tab:blue"]
plt.figure(figsize=(8, 3.5))
plt.barh(nombres, tiempos, color=colores); plt.xscale("log")
plt.xlabel("años (escala logarítmica)"); plt.title("¿Cuánto podría durar el Sol?"); plt.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## 3. El camino aleatorio de un fotón

Dentro del Sol, un fotón avanza una distancia pequeña, choca y sale en una dirección al azar. Tras $N$ pasos de longitud $\ell$, ¿a qué distancia del punto de partida está? No a $N\ell$, porque los pasos se anulan en parte. Simulamos muchos caminos en tres dimensiones y medimos la distancia típica (la raíz de la media de los cuadrados).

In [ ]:
rng = np.random.default_rng(34)
caminantes, pasos = 2000, 1000
direcciones = rng.normal(size=(caminantes, pasos, 3))
direcciones /= np.linalg.norm(direcciones, axis=2, keepdims=True)      # pasos de longitud 1 en dirección al azar
posiciones = np.cumsum(direcciones, axis=1)
distancia_rms = np.sqrt((np.linalg.norm(posiciones, axis=2) ** 2).mean(axis=0))

N = np.arange(1, pasos + 1)
plt.figure(figsize=(7, 4.5))
plt.loglog(N, distancia_rms, label="simulación")
plt.loglog(N, np.sqrt(N), "k--", label="√N")
plt.xlabel("número de pasos N"); plt.ylabel("distancia típica (en pasos)"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

ax = plt.figure(figsize=(6, 6)).add_subplot(projection="3d")
ax.plot(*posiciones[0].T, lw=0.6); ax.set_title("Un camino aleatorio de 1000 pasos"); plt.show()

La distancia crece como $\sqrt{N}$: para alejarse $D$ hacen falta $N = (D/\ell)^2$ pasos. En el Sol, con $D = R_\odot$ y un recorrido medio entre choques del orden del milímetro o menos (varía mucho con la profundidad), salen números enormes de pasos. El tiempo es $N\ell/c$.

In [ ]:
for ell in [1e-2, 1e-3, 1e-4]:          # recorrido libre medio en metros (valores ilustrativos)
    N_pasos = (R_sol / ell) ** 2
    t = N_pasos * ell / c / ANO
    anos = f"{t:,.0f}".replace(",", " ")
    print(f"ℓ = {ell * 1000:6.1f} mm → {N_pasos:.1e} pasos, unos {anos} años")

Según el valor que tomes, salen de miles a cientos de miles de años. Los modelos detallados, que tienen en cuenta cómo cambia la densidad con la profundidad, dan decenas o cientos de miles de años.

## 4. El equilibrio hidrostático de una esfera uniforme

Resolvemos numéricamente $dP/dr = -G M(r) \rho / r^2$ para una esfera de densidad constante, la más sencilla posible, integrando desde la superficie ($P = 0$) hacia el centro con pasos pequeños (como el método de Euler de la semana 11). Para este caso hay solución exacta, $P(r) = \tfrac{2\pi}{3}G\rho^2(R^2 - r^2)$, que sirve para comprobar.

In [ ]:
rho = M_sol / (4 / 3 * np.pi * R_sol**3)
r = np.linspace(R_sol, 1e3, 20000)          # de la superficie hacia el centro
dr = r[1] - r[0]                             # negativo
P = np.zeros_like(r)
for i in range(1, r.size):
    M_r = 4 / 3 * np.pi * r[i - 1] ** 3 * rho
    P[i] = P[i - 1] - G * M_r * rho / r[i - 1] ** 2 * dr
P_exacta = 2 * np.pi / 3 * G * rho**2 * (R_sol**2 - r**2)

plt.figure(figsize=(7, 4))
plt.plot(r / R_sol, P, label="numérica"); plt.plot(r / R_sol, P_exacta, "--", label="exacta")
plt.xlabel("r / R☉"); plt.ylabel("presión (Pa)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

T_c = P[-1] / (2 * rho / m_p * kB)           # gas ideal de protones y electrones
print(f"densidad media: {rho:.0f} kg/m³")
print(f"presión central: {P[-1]:.2e} Pa (unos {P[-1] / 1.013e5:.1e} atmósferas)")
print(f"temperatura central estimada: {T_c:.2e} K")

Con una densidad uniforme sale una temperatura central de unos 6 millones de grados. El Sol real está mucho más concentrado hacia el centro (allí la densidad es unas 100 veces la media), y su temperatura central es de unos 15 millones de K. El modelo es tosco, pero acierta el orden de magnitud: millones de grados, no miles ni miles de millones.

## 5. Experimenta

1. Calcula cuántos años de luminosidad actual daría la energía de la fusión de **todo** el hidrógeno del Sol (unos tres cuartos de su masa).
2. En el camino aleatorio, haz que los pasos sean más largos hacia fuera que hacia dentro (como si el gas empujara un poco la luz). ¿Cómo cambia la distancia frente a $N$?
3. En el equilibrio hidrostático, prueba una densidad que crezca hacia el centro, por ejemplo $\rho(r) = \rho_c\,(1 - r/R)$, y calcula $M(r)$ sumando capas. ¿Sube la temperatura central?

In [ ]:
# Tu experimento aquí